# 1. Install Requirements

In [1]:
!pip install -r requirements2.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.1/376.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.7/290.7 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 593.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 8.3 MB/s eta 0:00:00
   ━━━━━━

# 2. Import Libraries

In [2]:
import pandas as pd
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from tqdm import tqdm
import re
import numpy as np
import random
import huggingface_hub
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import zipfile


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# 3. Configuring Model (Load Model + Merge Lora Layer)

In [3]:
rank = 32
alpha = 32
dropout = 0.05

In [4]:
with zipfile.ZipFile(f"Komodo-7b-summarize-best-{rank}-{alpha}-{dropout}.zip", "r") as zip_ref:
    zip_ref.extractall()

In [5]:
huggingface_hub.login(HF_TOKEN)

In [6]:

# Load model dan tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Yellow-AI-NLP/komodo-7b-base",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
    token=True,
    trust_remote_code=True
)

# 3️⃣ Load adapter hasil fine-tuning
model = PeftModel.from_pretrained(model, f"./Komodo-7b-summarize-best-{rank}-{alpha}-{dropout}")  # path hasil fine-tuning
model = FastLanguageModel.for_inference(model)

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.9: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00006.safetensors:   0%|          | 0.00/4.89G [00:00<?, ?B/s]

model-00002-of-00006.safetensors:   0%|          | 0.00/4.86G [00:00<?, ?B/s]

model-00003-of-00006.safetensors:   0%|          | 0.00/4.86G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.39M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/58.1k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Yellow-AI-NLP/komodo-7b-base does not have a padding token! Will use pad_token = <unk>.


# 4. Configure Evaluation (BertScore + ROUGE)

In [7]:
# ======================================================
# 🔧 KONFIGURASI
# ======================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CONFIG = {
    "extractive": {
        "input_csv": "test_ext_sample_prompt.csv",
        "ref_col": "extractive_text_summary",
        "output_csv": f"testing_extractive_with_generated_summary_and_scores-{rank}-{alpha}-{dropout}.csv"
    },
    "abstractive": {
        "input_csv": "test_abs_sample_prompt.csv",
        "ref_col": "combined_clean_summary",
        "output_csv": f"testing_abstractive_with_generated_summary_and_scores-{rank}-{alpha}-{dropout}.csv"
    }
}

# ======================================================
# ⚙️ UTILITAS
# ======================================================
def extract_output(text: str) -> str:
    """Ambil isi teks antara [OUTPUT] dan [/OUTPUT]."""
    match = re.search(r"\[OUTPUT\](.*?)\[/OUTPUT\]", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    else:
        parts = re.split(r"\[OUTPUT\]", text)
        return parts[-1].strip() if len(parts) > 1 else text.strip()


def summarize(prompt_text: str, model, tokenizer) -> str:
    """Lakukan inferensi deterministik pada satu teks."""
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=2048).to(DEVICE)
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.0,
        do_sample=False
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return extract_output(decoded)


# ======================================================
# 🧠 PIPELINE UTAMA
# ======================================================
def run_evaluation(mode: str, model, tokenizer):
    """
    Jalankan seluruh pipeline inferensi dan evaluasi:
    1. Load data
    2. Inferensi model
    3. Hitung ROUGE dan BERTScore
    4. Simpan hasil
    """
    assert mode in CONFIG, f"Mode tidak valid: {mode}. Gunakan 'extractive' atau 'abstractive'."

    cfg = CONFIG[mode]
    df = pd.read_csv(cfg["input_csv"])
    print(f"📂 Mode: {mode.upper()} | Jumlah data uji: {len(df)} sampel")

    # Cek kolom wajib
    assert "text" in df.columns, "Kolom 'text' tidak ditemukan di CSV!"
    assert cfg["ref_col"] in df.columns, f"Kolom '{cfg['ref_col']}' tidak ditemukan di CSV!"

    # 🔹 Inferensi model
    print(f"\n🚀 Memulai inferensi model ({mode})...\n")
    generated_summaries = [
        summarize(text, model, tokenizer) for text in tqdm(df["text"], desc="Membuat ringkasan")
    ]
    df["generated_summary"] = generated_summaries

    # 🔹 Evaluasi ROUGE
    print("\n📏 Menghitung skor ROUGE...\n")
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rouge_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    for ref, gen in tqdm(zip(df[cfg["ref_col"]], df["generated_summary"]), total=len(df)):
        rouge = scorer.score(ref, gen)
        for k in rouge_scores.keys():
            rouge_scores[k].append(rouge[k].fmeasure)

    df["rouge1_f1"] = rouge_scores["rouge1"]
    df["rouge2_f1"] = rouge_scores["rouge2"]
    df["rougeL_f1"] = rouge_scores["rougeL"]

    # 🔹 Evaluasi BERTScore
    print("\n🤖 Menghitung BERTScore...\n")
    P, R, F1 = bert_score(
        cands=df["generated_summary"].tolist(),
        refs=df[cfg["ref_col"]].tolist(),
        lang="id",
        verbose=True,
        device=DEVICE
    )

    df["bertscore_precision"] = P.cpu().numpy()
    df["bertscore_recall"] = R.cpu().numpy()
    df["bertscore_f1"] = F1.cpu().numpy()

    # 🔹 Cetak hasil rata-rata
    print("\n📊 HASIL AKHIR (RATA-RATA F1):")
    print(f"ROUGE-1 : {np.mean(df['rouge1_f1']):.4f}")
    print(f"ROUGE-2 : {np.mean(df['rouge2_f1']):.4f}")
    print(f"ROUGE-L : {np.mean(df['rougeL_f1']):.4f}")
    print(f"BERTScore (F1): {np.mean(df['bertscore_f1']):.4f}")

    # 🔹 Simpan hasil
    df.to_csv(cfg["output_csv"], index=False)
    print(f"\n💾 Semua hasil disimpan di '{cfg['output_csv']}'")

In [8]:
run_evaluation("extractive", model, tokenizer)

📂 Mode: EXTRACTIVE | Jumlah data uji: 250 sampel

🚀 Memulai inferensi model (extractive)...



Membuat ringkasan: 100%|██████████| 250/250 [30:10<00:00,  7.24s/it]



📏 Menghitung skor ROUGE...



100%|██████████| 250/250 [00:00<00:00, 702.31it/s]



🤖 Menghitung BERTScore...



...

calculating scores...
computing bert embedding.


  0%|          | 0/8 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/4 [00:00<?, ?it/s]

done in 3.84 seconds, 65.08 sentences/sec

📊 HASIL AKHIR (RATA-RATA F1):
ROUGE-1 : 0.4914
ROUGE-2 : 0.4100
ROUGE-L : 0.4564
BERTScore (F1): 0.8011

💾 Semua hasil disimpan di 'testing_extractive_with_generated_summary_and_scores-32-32-0.05.csv'


In [9]:
run_evaluation("abstractive", model, tokenizer)

📂 Mode: ABSTRACTIVE | Jumlah data uji: 250 sampel

🚀 Memulai inferensi model (abstractive)...



Membuat ringkasan: 100%|██████████| 250/250 [30:02<00:00,  7.21s/it]



📏 Menghitung skor ROUGE...



100%|██████████| 250/250 [00:00<00:00, 962.36it/s]



🤖 Menghitung BERTScore...

calculating scores...
computing bert embedding.


  0%|          | 0/8 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/4 [00:00<?, ?it/s]

done in 1.63 seconds, 153.65 sentences/sec

📊 HASIL AKHIR (RATA-RATA F1):
ROUGE-1 : 0.4239
ROUGE-2 : 0.2416
ROUGE-L : 0.3526
BERTScore (F1): 0.7823

💾 Semua hasil disimpan di 'testing_abstractive_with_generated_summary_and_scores-32-32-0.05.csv'
